In [3]:
from collections import Counter
import numpy as np

from quantumsim.sparsedm import SparseDM as _SparseDM
from quantumsim import ptm as _ptm

import pygsti

import stim

from loqs.backends.reps import GateRep, InstrumentRep
from loqs.backends.circuit import ListPhysicalCircuit
from loqs.backends.model import DictNoiseModel
from loqs.backends.state import QSimQuantumState as QSimState
from loqs.backends.state import STIMQuantumState as STIMState
from loqs.core import QuantumProgram, QECCode
from loqs.core.instructions import builders
from loqs.tools import pygstitools as pt

In [4]:
p_depol = 0.1 # The chance we replace state with maximally mixed state
p_pauli = 3/4 * p_depol # This is the rate of each non-I Pauli being applied
p_flip = 2/4 * p_depol # This is the chance we 

In [5]:
# Greenbaum 1509.02921, Section 2.1.3 Example 1
ptm = np.array([
    [1, 0, 0, 0],
    [0, 1-p_depol, 0, 0],
    [0, 0, 1-p_depol, 0],
    [0, 0, 0, 1-p_depol]
])

In [6]:
qsim_ptm = pt.ptm_to_qsim_ptm(ptm)

qsim_native_ptm = _ptm.dephasing_ptm(p_depol, p_depol, p_depol)

# Double-check that these are the same
np.allclose(qsim_native_ptm, qsim_ptm)

True

In [7]:
# Test this numerically
depol_gate_dict = {("Gi", ("Q0",)): qsim_native_ptm} # Depolarizing idle
inst_dict = {("Iz", ("Q0",)): (0, True)} # Measure and reset
depol_noise_model = DictNoiseModel(
    (depol_gate_dict, inst_dict),
    gaterep=GateRep.QSIM_SUPEROPERATOR,
    instreps=[InstrumentRep.ZBASIS_PROJECTION]
)

In [8]:
_ptm.hadamard_ptm()

array([[ 0.5       ,  0.70710678,  0.        ,  0.5       ],
       [ 0.70710678,  0.        ,  0.        , -0.70710678],
       [ 0.        ,  0.        , -1.        ,  0.        ],
       [ 0.5       , -0.70710678,  0.        ,  0.5       ]])

In [9]:
# Make a dummy QECCode
Gi_Iz_instruction = builders.build_physical_circuit_instruction(
    ListPhysicalCircuit([[("Gi", ("Q0",))], [("Iz", ("Q0",))]]),
    name="Idle and measure circuit"
)

code_1Q = QECCode({"Gi + Iz": Gi_Iz_instruction}, ["Q0"], ["Q0"])

In [10]:
stack = [
    ("Init State", None, (1,), {"qubit_labels": ["Q0"]}),
    ("Init Patch 1Q", None, ("L0", ["Q0"])),
    ("Gi + Iz", "L0")
]

program = QuantumProgram(
    stack,
    default_noise_model=depol_noise_model,
    default_base_seed=20241104,
    state_type=QSimState,
    patch_types={"1Q": code_1Q},
    name="1Q depolarizing test"
)

In [11]:
program.run(num_shots=1000, reset_shot_histories=True)

Program 1Q depolarizing test: 100%|██████████| 1000/1000 [00:00<00:00, 1598.41it/s]


In [12]:
outs = [mo["Q0"][0] for mo in program.collect_shot_data("measurement_outcomes", -1)]

In [13]:
Counter(outs) == {0: 952, 1: 48}

True

In [14]:
# Test this numerically
stim_depol_gate_dict = {("Gi", ("Q0",)): f"DEPOLARIZE1({p_pauli}) 0"} # Depolarizing idle
inst_dict = {("Iz", ("Q0",)): (0, True)} # Measure and reset
stim_depol_noise_model = DictNoiseModel(
    (stim_depol_gate_dict, inst_dict),
    gaterep=GateRep.STIM_CIRCUIT_STR,
    instreps=[InstrumentRep.ZBASIS_PROJECTION]
)

In [15]:
program_stim = QuantumProgram.from_quantum_program(program, state_type=STIMState, default_noise_model=stim_depol_noise_model)

In [16]:
program_stim.run(num_shots=1000, reset_shot_histories=True)

Program 1Q depolarizing test: 100%|██████████| 1000/1000 [00:00<00:00, 3965.22it/s]


In [17]:
stim_outs = [mo["Q0"][0] for mo in program_stim.collect_shot_data("measurement_outcomes", -1)]

In [19]:
Counter(stim_outs)

Counter({0: 952, 1: 48})

In [57]:

rates = np.array([
#   I     X     Y     Z
    0   , 0.01, 0.02, 0.03,  # I
    0.01, 0.02, 0.03, 0.04,  # X
    0.04, 0.02, 0.03, 0.01,  # Y
    0.02, 0.01, 0.04, 0.03]) # Z


rates = np.array([
#   I     X     Y     Z
    0   , 0.05, 0.05, 0.00,  # I
    0.00, 0.00, 0.00, 0.00,  # X
    0.00, 0.00, 0.00, 0.00,  # Y
    0.00, 0.00, 0.00, 0.00]) # Z

# Probability of flipping just first bit is XI + YI
prob_10 = rates[4] + rates[8]

# Probability of flipping just second bit is IX + IY
prob_01 = rates[1] + rates[2]

# Probability of both flipping is XX + XY + YX + YY
prob_11 = rates[5] + rates[6] + rates[9] + rates[10]

prob_00 = 1 - prob_10 - prob_01 - prob_11
print(f"{prob_00=} {prob_10=} {prob_01=} {prob_11=}")

ptm_2q = np.eye(16)
for i, r in enumerate(rates):
    for j in range(16):
        if j in [0, i]:
            continue
        ptm_2q[j, j] -= r

print(np.array2string(ptm_2q, precision=2, suppress_small=True, max_line_width=200))

qsim_ptm_2q = pt.ptm_to_qsim_ptm(ptm_2q)

prob_00=0.9 prob_10=0.0 prob_01=0.1 prob_11=0.0
[[1.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.95 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.95 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.9  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.9  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.9  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.9  0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.9  0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.9  0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.9  0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.9  0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.

In [58]:
# Make a dummy QECCode
Gi_Iz_instruction = builders.build_physical_circuit_instruction(
    ListPhysicalCircuit([[("Gi", ("Q0", "Q1"))], [("Iz", ("Q0",)), ("Iz", ("Q1",))]]),
    name="Idle and measure circuit"
)

code_2Q = QECCode({"Gi + Iz": Gi_Iz_instruction}, ["Q0", "Q1"], ["Q0", "Q1"])

In [59]:
depol2Q_gate_dict = {("Gi", ("Q0", "Q1")): qsim_ptm_2q} # Depolarizing idle
inst_dict = {("Iz", ("Q0",)): (0, True), ("Iz", ("Q1",)): (0, True)} # Measure and reset
depol2Q_noise_model = DictNoiseModel(
    (depol2Q_gate_dict, inst_dict),
    gaterep=GateRep.QSIM_SUPEROPERATOR,
    instreps=[InstrumentRep.ZBASIS_PROJECTION]
)

In [60]:
stack = [
    ("Init State", None, (2,), {"qubit_labels": ["Q0", "Q1"]}),
    ("Init Patch 2Q", None, ("L0", ["Q0", "Q1"])),
    ("Gi + Iz", "L0")
]

program = QuantumProgram(
    stack,
    default_noise_model=depol2Q_noise_model,
    default_base_seed=20241104,
    state_type=QSimState,
    patch_types={"2Q": code_2Q},
    name="2Q Pauli channel test"
)

In [61]:
program.run(10000, reset_shot_histories=True)

Program 2Q Pauli channel test: 100%|██████████| 10000/10000 [00:07<00:00, 1256.98it/s]


In [62]:
outs = ["".join([str(mo[q][0]) for q in ["Q0", "Q1"]]) for mo in program.collect_shot_data("measurement_outcomes", -1)]

In [63]:
Counter(outs)

Counter({'00': 9302, '01': 245, '11': 238, '10': 215})